# Risk Premia Pairs PCA Backtest

Research notebook for `BT.signals.risk_premia_pairs_pca_backtest`.

The production module does the strategy work: PCA residual signal generation, Phase-1 admission, Phase-2 shrinkage-covariance sizing, risk overlay, and `QueryDrivenBacktest` order flow. This notebook is the analysis harness around it: panel diagnostics, signal diagnostics, proxy PnL, query backtest tear sheets, trade attribution, and parameter sweeps.

Default mode is a deterministic synthetic panel so every analytics section is runnable without live market data. Switch `PARAMS["data_mode"]` to `"csv"` or `"swap_spread_tb"` for real research panels.

## Technical Context

The attached technical appendix maps cleanly into this notebook in four layers:

1. PCA decomposes a rates or swap-spread panel into orthogonal common factors. Residuals are the piece not explained by the retained factors.
2. Large residual z-scores are treated as rich/cheap dislocations. The trade direction is mean reversion: short expensive positive residuals, long cheap negative residuals.
3. Phase-1 admits ideas with enough realized directed Sharpe. Phase-2 sizes the admitted ideas with a shrunk covariance estimate and an L2 pull back to Phase-1 weights.
4. Volatility/risk overlay is separate from alpha generation. The signal table decides what belongs in the book; the overlay scales the book.

For rates RV this is closer to the PCA dislocation framework than a yield-as-expected-return asset-allocation model. Swap spreads, butterflies, basis packages, or forward-rate building blocks can all be supplied as the input panel as long as each column maps to an executable idea id.

## 1. Parameters

In [ ]:
# Central controls. Change these before running the notebook.
PARAMS = {
    # Data modes: "synthetic", "csv", or "swap_spread_tb".
    "data_mode": "synthetic",
    "csv_panel_path": None,
    "csv_date_column": None,

    # Synthetic panel controls.
    "synthetic_start": "2023-01-02",
    "synthetic_periods": 640,
    "synthetic_seed": 7,
    "synthetic_ideas": [
        "2Y",
        "5Y",
        "10Y",
        "30Y",
        "2Y/5Y/10Y",
        "5Y/10Y/30Y",
        "2Y/10Y",
        "5Y/30Y",
    ],

    # Optional live/intraday swap-spread panel controls.
    "curve": "USD-SOFR-1D",
    "swap_spread_source": "SDR-USTS_WEBULL_WSJ_LIVE-INTRADAY-SPREADOVER",
    "data_start": "2026-03-20 07:00",
    "data_end": "2026-03-24 15:00",
    "swap_spread_tenors": ["2Y", "5Y", "10Y", "30Y"],
    "resample_rule": "1D",
    "smoothing_window": 15,

    # Backtest execution controls. Synthetic mode has a demo MDP; live modes require a real mdp/query_factory.
    "run_query_backtest": True,
    "run_query_tearsheet": True,
    "run_param_sweep": False,
    "export_outputs": False,
    "show_progress": False,

    # Strategy controls passed to RiskPremiaPairsPCAConfig.
    "config": {
        "curve": "USD-SOFR-1D",
        "pca_window_days": 120,
        "pca_components": 3,
        "residual_z_entry": 1.25,
        "residual_z_exit": 0.30,
        "residual_z_stop": 3.00,
        "phase1_lookback_days": 63,
        "phase1_min_realized_sharpe": -0.50,
        "require_momentum_confirmation": False,
        "require_positioning_confirmation": False,
        "covariance_lookback_days": 63,
        "risk_aversion": 1.0,
        "l2_to_phase1": 10.0,
        "max_abs_weight": 0.35,
        "min_abs_target_weight": 0.02,
        "gross_leverage": 1.0,
        "target_annual_vol": None,
        "min_overlay_scale_to_trade": 0.05,
        "trade_bpv": 100_000.0,
        "max_concurrent_positions": 6,
        "max_holding_days": 22,
        "no_duplicate_ideas": True,
        "exit_when_signal_missing": False,
        "round_trip_cost_bp": 0.0,
    },
}


## 2. Imports And Repo Setup

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import datetime as dt
import itertools
import os
import re
import sys
import warnings
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    HAS_PLOTLY = True
except Exception:
    HAS_PLOTLY = False

from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.options.display.max_columns = 80
pd.options.display.width = 160
pd.options.display.float_format = "{:,.4f}".format
sns.set_theme(style="whitegrid", context="notebook")

# ARBS local defaults. Keep notebook runs local unless a live provider is explicitly used.
os.environ.setdefault("ARBS_SUPABASE_ENABLED", "0")

REPO_ROOT = Path.cwd()
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "BT").exists() and (candidate / "Query").exists():
        REPO_ROOT = candidate
        break

os.chdir(REPO_ROOT)
os.environ.setdefault("ARBS_CACHE_DIR", str(REPO_ROOT / ".cache"))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from BT.signals.risk_premia_pairs_pca_backtest import (
    RiskPremiaPairsPCAConfig,
    RiskPremiaPCASignal,
    RiskPremiaPCASnapshot,
    build_risk_premia_pca_signal_table,
    make_irs_query_factory,
    run_risk_premia_pairs_pca_backtest,
)
from BT.signals.risk_premia_pairs_pca_backtest import _pca_residual_snapshot

try:
    from BT.query_tearsheet import create_query_backtest_tearsheet
except Exception:
    create_query_backtest_tearsheet = None

print(f"Repo root: {REPO_ROOT}")
print(f"ARBS cache: {os.environ.get('ARBS_CACHE_DIR')}")
print(f"Plotly available: {HAS_PLOTLY}")


## 3. Build Configuration

In [ ]:
config = RiskPremiaPairsPCAConfig(**PARAMS["config"])
config_frame = pd.Series(asdict(config), name="value").to_frame()
display(config_frame)


## 4. Load Or Generate The Signal Panel

`panel` is date x idea-id levels in bps or level-like units. For swap-spread RV, columns are spread tickers, butterflies, curve packages, or related rates instruments. `returns_panel` is the PnL/return proxy used for Phase-1 Sharpe and Phase-2 covariance. If omitted, the module uses `panel.diff()`.

In [ ]:
def _tenor_numbers(label: str) -> list[float]:
    return [float(x) for x in re.findall(r"(\d+(?:\.\d+)?)Y", str(label).upper())]


def make_synthetic_pca_panel(
    *,
    start: str,
    periods: int,
    ideas: list[str],
    seed: int = 7,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Create a deterministic panel with common PCA factors plus staged dislocations."""
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range(start=start, periods=periods)
    t = np.arange(periods, dtype=float)

    level = np.cumsum(rng.normal(0.0, 0.045, periods))
    slope = np.cumsum(rng.normal(0.0, 0.035, periods))
    curvature = np.cumsum(rng.normal(0.0, 0.030, periods))
    carry = 0.80 * np.sin(t / 42.0) + 0.35 * np.cos(t / 91.0)

    panel = pd.DataFrame(index=dates, columns=ideas, dtype=float)
    for idea in ideas:
        tenors = _tenor_numbers(idea)
        avg_tenor = np.mean(tenors) if tenors else 7.0
        span = (max(tenors) - min(tenors)) if len(tenors) >= 2 else 0.0
        slash_count = str(idea).count("/")
        noise = rng.normal(0.0, 0.12 + 0.02 * slash_count, periods)

        if slash_count == 0:
            series = (
                15.0
                + 0.55 * level
                + (avg_tenor / 30.0) * slope
                + (0.10 + avg_tenor / 100.0) * curvature
                + 0.15 * carry
                + noise
            )
        elif slash_count == 1:
            series = (
                2.0
                + 0.20 * level
                + (span / 25.0) * slope
                + 0.40 * curvature
                + 0.20 * carry
                + noise
            )
        else:
            series = (
                0.5
                + 0.05 * level
                + 0.15 * slope
                + (0.80 + span / 25.0) * curvature
                + 0.30 * carry
                + noise
            )
        panel[idea] = series

    # Staged dislocations: build, then partially mean-revert, so entry and exit analytics have something to inspect.
    dislocations = [
        ("5Y/10Y/30Y", 0.58, 0.80, 4.50),
        ("2Y/5Y/10Y", 0.66, 0.90, -3.50),
        ("10Y", 0.72, 0.92, 2.25),
        ("2Y/10Y", 0.76, 0.94, -1.75),
    ]
    for idea, build_start, revert_start, magnitude in dislocations:
        if idea not in panel:
            continue
        effect = np.zeros(periods)
        b0 = int(periods * build_start)
        r0 = int(periods * revert_start)
        effect[b0:r0] = np.linspace(0.0, magnitude, max(r0 - b0, 1))
        effect[r0:] = np.linspace(magnitude, magnitude * 0.15, max(periods - r0, 1))
        panel[idea] += effect

    returns_panel = panel.diff()
    momentum_panel = panel.diff(21)
    positioning_panel = -panel.rolling(126, min_periods=20).apply(
        lambda x: (x[-1] - np.nanmean(x)) / (np.nanstd(x, ddof=1) or np.nan),
        raw=True,
    )
    return panel, returns_panel, momentum_panel, positioning_panel


def load_csv_panel(path: str | Path, *, date_column: str | None = None) -> pd.DataFrame:
    raw = pd.read_csv(path)
    if date_column is None:
        date_column = raw.columns[0]
    out = raw.set_index(date_column)
    out.index = pd.to_datetime(out.index)
    return out.sort_index().apply(pd.to_numeric, errors="coerce")


def load_swap_spread_tb_panel(params: dict[str, Any]) -> pd.DataFrame:
    from TB.IRSwapSpreadsTB import IRSwapSpreadsTB

    tb = IRSwapSpreadsTB(
        source=params["swap_spread_source"],
        curve_name=params["curve"],
        show_tqdm=params["show_progress"],
    )
    raw = tb.get_timeseries(
        start=pd.Timestamp(params["data_start"]).to_pydatetime(),
        end=pd.Timestamp(params["data_end"]).to_pydatetime(),
        tenors=params["swap_spread_tenors"],
        smoothing_window=params["smoothing_window"],
    )

    cols = {}
    for tenor in params["swap_spread_tenors"]:
        step_col = f"{tenor}_SWAP_SPREAD_BPS_STEP"
        raw_col = f"{tenor}_SWAP_SPREAD_BPS"
        if step_col in raw.columns:
            cols[step_col] = tenor
        elif raw_col in raw.columns:
            cols[raw_col] = tenor

    panel = raw[list(cols)].rename(columns=cols).apply(pd.to_numeric, errors="coerce")
    if params.get("resample_rule"):
        panel = panel.resample(params["resample_rule"]).last().dropna(how="all")
    return panel


data_mode = PARAMS["data_mode"].lower()
if data_mode == "synthetic":
    panel, returns_panel, momentum_panel, positioning_panel = make_synthetic_pca_panel(
        start=PARAMS["synthetic_start"],
        periods=PARAMS["synthetic_periods"],
        ideas=PARAMS["synthetic_ideas"],
        seed=PARAMS["synthetic_seed"],
    )
elif data_mode == "csv":
    if not PARAMS["csv_panel_path"]:
        raise ValueError("Set PARAMS['csv_panel_path'] before using data_mode='csv'.")
    panel = load_csv_panel(PARAMS["csv_panel_path"], date_column=PARAMS["csv_date_column"])
    returns_panel = panel.diff()
    momentum_panel = panel.diff(21)
    positioning_panel = None
elif data_mode == "swap_spread_tb":
    panel = load_swap_spread_tb_panel(PARAMS)
    returns_panel = panel.diff()
    momentum_panel = panel.diff(21)
    positioning_panel = None
else:
    raise ValueError(f"Unsupported data_mode={PARAMS['data_mode']!r}")

panel = panel.sort_index().apply(pd.to_numeric, errors="coerce")
returns_panel = returns_panel.reindex(index=panel.index, columns=panel.columns)
if momentum_panel is not None:
    momentum_panel = momentum_panel.reindex(index=panel.index, columns=panel.columns)
if positioning_panel is not None:
    positioning_panel = positioning_panel.reindex(index=panel.index, columns=panel.columns)

print(f"Panel shape: {panel.shape[0]:,} dates x {panel.shape[1]:,} ideas")
display(panel.tail())


## 5. Notebook Analytics Helpers

In [ ]:
def compute_pca_diagnostic_panels(
    panel: pd.DataFrame,
    config: RiskPremiaPairsPCAConfig,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    levels = panel.sort_index().apply(pd.to_numeric, errors="coerce")
    min_history = max(5, min(int(config.pca_window_days), len(levels)))
    residual_rows = []
    zscore_rows = []
    evr_rows = []

    for loc in range(min_history - 1, len(levels)):
        ts = pd.Timestamp(levels.index[loc])
        window = levels.iloc[max(0, loc - int(config.pca_window_days) + 1) : loc + 1]
        residuals, zscores, diag = _pca_residual_snapshot(window, config.pca_components)
        residual_rows.append(residuals.rename(ts))
        zscore_rows.append(zscores.rename(ts))
        evr = diag.get("explained_variance_ratio", []) or []
        evr_rows.append(
            {
                "timestamp": ts,
                "n_components": diag.get("n_components", np.nan),
                "explained_total": float(np.sum(evr)) if len(evr) else np.nan,
                **{f"pc{i + 1}": float(v) for i, v in enumerate(evr)},
            }
        )

    residual_panel = pd.DataFrame(residual_rows).sort_index()
    zscore_panel = pd.DataFrame(zscore_rows).sort_index()
    evr_frame = pd.DataFrame(evr_rows).set_index("timestamp").sort_index()
    return residual_panel, zscore_panel, evr_frame


def signal_table_to_frames(
    signal_table: dict[pd.Timestamp, RiskPremiaPCASnapshot],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    signal_rows = []
    snapshot_rows = []
    for ts, snapshot in sorted(signal_table.items(), key=lambda item: pd.Timestamp(item[0])):
        diag = snapshot.diagnostics or {}
        evr = diag.get("explained_variance_ratio", []) or []
        snapshot_rows.append(
            {
                "timestamp": pd.Timestamp(ts),
                "n_signals": len(snapshot.signals),
                "overlay_scale": snapshot.overlay_scale,
                "annual_vol": diag.get("annual_vol", np.nan),
                "n_pca_columns": len(diag.get("pca_columns", []) or []),
                "explained_total": float(np.sum(evr)) if len(evr) else np.nan,
                "pc1": float(evr[0]) if len(evr) > 0 else np.nan,
                "pc2": float(evr[1]) if len(evr) > 1 else np.nan,
                "pc3": float(evr[2]) if len(evr) > 2 else np.nan,
                "reason": diag.get("reason"),
            }
        )
        for signal in snapshot.signals:
            signal_rows.append(
                {
                    "timestamp": signal.timestamp,
                    "idea_id": signal.idea_id,
                    "residual": signal.residual,
                    "zscore": signal.zscore,
                    "direction": signal.direction,
                    "phase1_sharpe": signal.phase1_sharpe,
                    "phase1_weight": signal.phase1_weight,
                    "target_weight": signal.target_weight,
                    "overlay_scale": signal.overlay_scale,
                    "expected_edge_bp": signal.expected_edge_bp,
                    "covariance_vol": signal.covariance_vol,
                    "rank_score": signal.rank_score,
                    "abs_target_weight": abs(signal.target_weight),
                    "abs_zscore": abs(signal.zscore),
                }
            )

    signals_df = pd.DataFrame(signal_rows)
    if not signals_df.empty:
        signals_df = signals_df.sort_values(["timestamp", "abs_target_weight"], ascending=[True, False]).reset_index(drop=True)
    snapshots_df = pd.DataFrame(snapshot_rows).set_index("timestamp").sort_index()
    return signals_df, snapshots_df


def build_weight_panel(signals_df: pd.DataFrame, index: pd.Index, columns: pd.Index) -> pd.DataFrame:
    if signals_df.empty:
        return pd.DataFrame(0.0, index=index, columns=columns)
    weights = signals_df.pivot_table(index="timestamp", columns="idea_id", values="target_weight", aggfunc="sum")
    return weights.reindex(index=index, columns=columns).fillna(0.0)


def signal_proxy_pnl(weights: pd.DataFrame, returns_panel: pd.DataFrame) -> pd.Series:
    aligned_returns = returns_panel.reindex(index=weights.index, columns=weights.columns).fillna(0.0)
    return (weights.shift(1).fillna(0.0) * aligned_returns).sum(axis=1).rename("signal_proxy_pnl")


def performance_stats(pnl: pd.Series, *, annualization: float = 252.0) -> pd.Series:
    clean = pd.to_numeric(pnl, errors="coerce").dropna()
    if clean.empty:
        return pd.Series(dtype=float)
    equity = clean.cumsum()
    drawdown = equity - equity.cummax()
    downside = clean[clean < 0.0]
    ann_mean = clean.mean() * annualization
    ann_vol = clean.std(ddof=1) * np.sqrt(annualization) if len(clean) > 1 else np.nan
    downside_vol = downside.std(ddof=1) * np.sqrt(annualization) if len(downside) > 1 else np.nan
    gross_profit = clean[clean > 0.0].sum()
    gross_loss = -clean[clean < 0.0].sum()
    return pd.Series(
        {
            "observations": len(clean),
            "total_pnl": clean.sum(),
            "annualized_mean": ann_mean,
            "annualized_vol": ann_vol,
            "sharpe": ann_mean / ann_vol if ann_vol and np.isfinite(ann_vol) and ann_vol > 0 else np.nan,
            "sortino": ann_mean / downside_vol if downside_vol and np.isfinite(downside_vol) and downside_vol > 0 else np.nan,
            "hit_rate": (clean > 0.0).mean(),
            "avg_win": clean[clean > 0.0].mean() if (clean > 0.0).any() else np.nan,
            "avg_loss": clean[clean < 0.0].mean() if (clean < 0.0).any() else np.nan,
            "profit_factor": gross_profit / gross_loss if gross_loss > 0 else np.nan,
            "max_drawdown": drawdown.min(),
            "calmar_like": ann_mean / abs(drawdown.min()) if drawdown.min() < 0 else np.nan,
        }
    )


def monthly_pnl_table(pnl: pd.Series) -> pd.DataFrame:
    clean = pd.to_numeric(pnl, errors="coerce").dropna()
    if clean.empty:
        return pd.DataFrame()
    monthly = clean.resample("ME").sum()
    table = monthly.to_frame("pnl")
    table["year"] = table.index.year
    table["month"] = table.index.strftime("%b")
    return table.pivot(index="year", columns="month", values="pnl").reindex(
        columns=["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    )


def drawdown_episodes(pnl: pd.Series, top_n: int = 10) -> pd.DataFrame:
    clean = pd.to_numeric(pnl, errors="coerce").dropna()
    if clean.empty:
        return pd.DataFrame(columns=["start", "trough", "end", "drawdown", "duration_days"])
    equity = clean.cumsum()
    running_max = equity.cummax()
    dd = equity - running_max
    episodes = []
    in_dd = False
    start = trough = None
    trough_val = 0.0
    for ts, value in dd.items():
        if value < 0 and not in_dd:
            in_dd = True
            start = ts
            trough = ts
            trough_val = value
        elif value < 0 and in_dd and value < trough_val:
            trough = ts
            trough_val = value
        elif value >= 0 and in_dd:
            episodes.append({"start": start, "trough": trough, "end": ts, "drawdown": trough_val})
            in_dd = False
    if in_dd:
        episodes.append({"start": start, "trough": trough, "end": clean.index[-1], "drawdown": trough_val})
    out = pd.DataFrame(episodes)
    if out.empty:
        return out
    out["duration_days"] = (pd.to_datetime(out["end"]) - pd.to_datetime(out["start"])).dt.days
    return out.sort_values("drawdown").head(top_n).reset_index(drop=True)


## 6. Build Signals

In [ ]:
signal_table = build_risk_premia_pca_signal_table(
    panel=panel,
    config=config,
    returns_panel=returns_panel,
    momentum_panel=momentum_panel,
    positioning_panel=positioning_panel,
)

residual_panel, zscore_panel, pca_evr = compute_pca_diagnostic_panels(panel, config)
signals_df, snapshots_df = signal_table_to_frames(signal_table)
weights_panel = build_weight_panel(signals_df, panel.index, panel.columns)
proxy_pnl = signal_proxy_pnl(weights_panel, returns_panel)

print(f"Snapshots: {len(signal_table):,}")
print(f"Signal rows: {len(signals_df):,}")
display(snapshots_df.tail())
display(signals_df.tail(20) if not signals_df.empty else pd.DataFrame())


## 7. Input Panel And PCA Diagnostics

In [ ]:
def plot_input_panel(panel: pd.DataFrame, returns_panel: pd.DataFrame, lookback: int = 252):
    fig, axes = plt.subplots(2, 2, figsize=(18, 10))
    tail = panel.tail(lookback)
    tail.plot(ax=axes[0, 0], lw=1.4)
    axes[0, 0].set_title(f"Panel Levels - Last {min(lookback, len(panel))} Observations")
    axes[0, 0].set_xlabel("")
    axes[0, 0].legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)

    returns_panel.tail(lookback).cumsum().plot(ax=axes[0, 1], lw=1.2)
    axes[0, 1].set_title("Cumulative Return/PnL Proxy By Idea")
    axes[0, 1].set_xlabel("")
    axes[0, 1].legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)

    corr = returns_panel.tail(lookback).corr()
    sns.heatmap(corr, ax=axes[1, 0], cmap="vlag", center=0.0, vmin=-1.0, vmax=1.0, annot=False)
    axes[1, 0].set_title("Return Proxy Correlation")

    if not pca_evr.empty:
        pca_evr[[c for c in ["pc1", "pc2", "pc3", "explained_total"] if c in pca_evr]].tail(lookback).plot(ax=axes[1, 1], lw=1.4)
        axes[1, 1].set_title("Rolling PCA Explained Variance")
        axes[1, 1].set_xlabel("")
    else:
        axes[1, 1].set_axis_off()

    fig.tight_layout()
    return fig

plot_input_panel(panel, returns_panel)


## 8. Signal Funnel And Latest Snapshot

In [ ]:
def plot_signal_diagnostics(
    signals_df: pd.DataFrame,
    snapshots_df: pd.DataFrame,
    zscore_panel: pd.DataFrame,
    weights_panel: pd.DataFrame,
):
    fig, axes = plt.subplots(3, 2, figsize=(18, 13))

    snapshots_df["n_signals"].plot(ax=axes[0, 0], lw=1.4, color="tab:blue")
    axes[0, 0].set_title("Signal Count Through Time")
    axes[0, 0].set_xlabel("")

    cols = [c for c in ["overlay_scale", "annual_vol"] if c in snapshots_df.columns]
    if cols:
        snapshots_df[cols].plot(ax=axes[0, 1], lw=1.4)
        axes[0, 1].set_title("Risk Overlay And Phase-2 Annual Vol")
        axes[0, 1].set_xlabel("")
    else:
        axes[0, 1].set_axis_off()

    if not zscore_panel.empty:
        latest_z = zscore_panel.iloc[-1].dropna().sort_values()
        colors = ["tab:red" if value > 0 else "tab:green" for value in latest_z]
        latest_z.plot(kind="barh", ax=axes[1, 0], color=colors)
        axes[1, 0].axvline(config.residual_z_entry, color="black", lw=1, ls="--")
        axes[1, 0].axvline(-config.residual_z_entry, color="black", lw=1, ls="--")
        axes[1, 0].set_title("Latest PCA Residual Z-Score")
    else:
        axes[1, 0].set_axis_off()

    if not signals_df.empty:
        counts = signals_df["idea_id"].value_counts().head(20)
        counts.plot(kind="bar", ax=axes[1, 1], color="tab:purple")
        axes[1, 1].set_title("Most Frequently Admitted Ideas")
        axes[1, 1].tick_params(axis="x", rotation=45)
    else:
        axes[1, 1].set_axis_off()

    if not signals_df.empty:
        axes[2, 0].scatter(signals_df["phase1_weight"], signals_df["target_weight"], alpha=0.55, s=24)
        lim = np.nanmax(np.abs(signals_df[["phase1_weight", "target_weight"]].to_numpy()))
        lim = max(float(lim), 0.05) if np.isfinite(lim) else 0.5
        axes[2, 0].plot([-lim, lim], [-lim, lim], color="black", lw=1, ls="--")
        axes[2, 0].set_xlim(-lim, lim)
        axes[2, 0].set_ylim(-lim, lim)
        axes[2, 0].set_title("Phase-1 Prior Weight Vs Phase-2 Target Weight")
        axes[2, 0].set_xlabel("Phase-1 Weight")
        axes[2, 0].set_ylabel("Phase-2 Target Weight")
    else:
        axes[2, 0].set_axis_off()

    gross = weights_panel.abs().sum(axis=1)
    net = weights_panel.sum(axis=1)
    pd.DataFrame({"gross": gross, "net": net}).plot(ax=axes[2, 1], lw=1.4)
    axes[2, 1].set_title("Signal-Level Gross And Net Target Weight")
    axes[2, 1].set_xlabel("")

    fig.tight_layout()
    return fig

plot_signal_diagnostics(signals_df, snapshots_df, zscore_panel, weights_panel)

latest_date = snapshots_df.index[-1] if len(snapshots_df) else None
if latest_date is not None:
    display(Markdown(f"### Latest Snapshot: {pd.Timestamp(latest_date).date()}"))
    latest = signals_df.loc[signals_df["timestamp"].eq(latest_date)].copy() if not signals_df.empty else pd.DataFrame()
    display(latest if not latest.empty else pd.DataFrame(columns=signals_df.columns if not signals_df.empty else []))


## 9. Residual, Z-Score, And Weight Heatmaps

In [ ]:
def plot_signal_heatmaps(
    zscore_panel: pd.DataFrame,
    residual_panel: pd.DataFrame,
    weights_panel: pd.DataFrame,
    lookback: int = 126,
):
    fig, axes = plt.subplots(3, 1, figsize=(18, 13), sharex=True)

    if not zscore_panel.empty:
        sns.heatmap(
            zscore_panel.tail(lookback).T,
            ax=axes[0],
            cmap="vlag",
            center=0.0,
            cbar_kws={"label": "z-score"},
        )
        axes[0].set_title(f"PCA Residual Z-Scores - Last {min(lookback, len(zscore_panel))} Observations")
    else:
        axes[0].set_axis_off()

    if not residual_panel.empty:
        sns.heatmap(
            residual_panel.tail(lookback).T,
            ax=axes[1],
            cmap="vlag",
            center=0.0,
            cbar_kws={"label": "residual"},
        )
        axes[1].set_title("PCA Residuals")
    else:
        axes[1].set_axis_off()

    sns.heatmap(
        weights_panel.tail(lookback).T,
        ax=axes[2],
        cmap="vlag",
        center=0.0,
        cbar_kws={"label": "target weight"},
    )
    axes[2].set_title("Target Weights After Phase-2 And Overlay")
    axes[2].set_xlabel("Date")
    fig.tight_layout()
    return fig

plot_signal_heatmaps(zscore_panel, residual_panel, weights_panel)


## 10. Signal-Level Proxy PnL

This is not the executed query backtest. It is a fast diagnostic that applies yesterday's target weights to today's `returns_panel`, useful for parameter sweeps and sanity checks before expensive query execution.

In [ ]:
proxy_stats = performance_stats(proxy_pnl)
display(proxy_stats.to_frame("signal_proxy"))
display(drawdown_episodes(proxy_pnl))

def plot_performance_dashboard(pnl: pd.Series, *, title: str = "Performance", annualization: float = 252.0):
    clean = pd.to_numeric(pnl, errors="coerce").dropna()
    fig, axes = plt.subplots(3, 2, figsize=(18, 12))
    if clean.empty:
        for ax in axes.ravel():
            ax.set_axis_off()
        return fig

    equity = clean.cumsum()
    drawdown = equity - equity.cummax()
    rolling_window = min(63, max(5, len(clean) // 3))
    rolling_mean = clean.rolling(rolling_window).mean() * annualization
    rolling_vol = clean.rolling(rolling_window).std() * np.sqrt(annualization)
    rolling_sharpe = rolling_mean / rolling_vol.replace(0.0, np.nan)

    equity.plot(ax=axes[0, 0], lw=1.6, color="tab:blue")
    axes[0, 0].set_title(f"{title}: Cumulative PnL")
    axes[0, 0].set_xlabel("")

    drawdown.plot(ax=axes[0, 1], lw=1.4, color="tab:red")
    axes[0, 1].fill_between(drawdown.index, drawdown.to_numpy(), 0.0, color="tab:red", alpha=0.20)
    axes[0, 1].set_title("Drawdown")
    axes[0, 1].set_xlabel("")

    clean.plot(ax=axes[1, 0], kind="hist", bins=50, color="tab:gray", alpha=0.80)
    axes[1, 0].axvline(clean.mean(), color="black", lw=1.2, ls="--")
    axes[1, 0].set_title("Daily PnL Distribution")

    rolling_sharpe.plot(ax=axes[1, 1], lw=1.4, color="tab:green")
    axes[1, 1].axhline(0.0, color="black", lw=1)
    axes[1, 1].set_title(f"Rolling Sharpe ({rolling_window} obs)")
    axes[1, 1].set_xlabel("")

    monthly = monthly_pnl_table(clean)
    if not monthly.empty:
        sns.heatmap(monthly, ax=axes[2, 0], cmap="vlag", center=0.0, annot=True, fmt=".2f")
        axes[2, 0].set_title("Monthly PnL")
    else:
        axes[2, 0].set_axis_off()

    contrib = (weights_panel.shift(1).fillna(0.0) * returns_panel.reindex_like(weights_panel).fillna(0.0)).sum().sort_values()
    contrib.plot(kind="barh", ax=axes[2, 1], color=["tab:red" if x < 0 else "tab:green" for x in contrib])
    axes[2, 1].set_title("Signal Proxy Contribution By Idea")

    fig.tight_layout()
    return fig

plot_performance_dashboard(proxy_pnl, title="Signal Proxy")


## 11. Query Backtest Execution

Synthetic mode includes a small deterministic MDP compatible with `make_irs_query_factory`, so the notebook can demonstrate actual `QueryDrivenBacktest` order flow. For live data modes, fill in `execution_mdp` and `execution_query_factory` with the actual package mapping you want traded before setting `run_query_backtest=True`.

In [ ]:
@dataclass(frozen=True)
class _DemoSwap:
    tenor: str
    notional: float
    fixed_rate: float = 0.0

    def with_notional(self, notional: float) -> "_DemoSwap":
        return _DemoSwap(tenor=self.tenor, notional=float(notional), fixed_rate=self.fixed_rate)


class _DemoCurve:
    def __init__(self, as_of_date: dt.date):
        self.as_of_date = as_of_date
        self.day_index = max(0, len(pd.bdate_range(dt.date(2023, 1, 2), as_of_date)) - 1)

    def id(self) -> str:
        return PARAMS["curve"]

    def reference_date(self) -> dt.date:
        return self.as_of_date

    def calendar_advance(self, ref_date, tenor):
        return ref_date

    def build_irswap(
        self,
        fwd=None,
        tenor=None,
        effective_date=None,
        maturity_date=None,
        fixed_rate=-0.0,
        notional=None,
        bpv=None,
    ) -> _DemoSwap:
        _ = fwd, effective_date, maturity_date
        tenor_label = tenor or "1Y"
        tenor_years = float(str(tenor_label).upper().replace("Y", ""))
        if notional is None:
            notional = float(bpv) / max(tenor_years * 1e-4, 1e-12) if bpv is not None else 1_000_000.0
        return _DemoSwap(tenor=tenor_label, notional=float(notional), fixed_rate=float(fixed_rate or 0.0))

    def fair_rate(self, instrument: _DemoSwap) -> float:
        tenor_years = float(str(instrument.tenor).upper().replace("Y", ""))
        cyclical = 0.00025 * np.sin(self.day_index / 35.0 + tenor_years / 10.0)
        drift = self.day_index * 0.000002 * tenor_years
        return 0.025 + tenor_years * 0.00025 + cyclical + drift

    def npv(self, instrument: _DemoSwap) -> float:
        return float(instrument.notional) * (self.fair_rate(instrument) - float(instrument.fixed_rate))

    def pv01(self, instrument: _DemoSwap) -> float:
        tenor_years = float(str(instrument.tenor).upper().replace("Y", ""))
        return float(instrument.notional) * tenor_years * 1e-4

    def resolve_pricable(self, pricable: _DemoSwap, risk_weight: float = 1.0) -> _DemoSwap:
        return pricable.with_notional(pricable.notional * float(risk_weight))


class _DemoSwapMDP:
    def get_pricer(self, request):
        ts = request["timestamp"]
        if isinstance(ts, dt.datetime):
            ts = ts.date()
        return _DemoCurve(ts)


result = None
if PARAMS["run_query_backtest"]:
    if data_mode == "synthetic":
        result = run_risk_premia_pairs_pca_backtest(
            signal_table=signal_table,
            mdp=_DemoSwapMDP(),
            query_factory=make_irs_query_factory(curve=config.curve, default_bpv=config.trade_bpv),
            config=config,
            show_progress=PARAMS["show_progress"],
        )
    else:
        execution_mdp = None
        execution_query_factory = None
        if execution_mdp is None or execution_query_factory is None:
            print("Live query backtest skipped. Set execution_mdp and execution_query_factory in this cell.")
        else:
            result = run_risk_premia_pairs_pca_backtest(
                signal_table=signal_table,
                mdp=execution_mdp,
                query_factory=execution_query_factory,
                config=config,
                show_progress=PARAMS["show_progress"],
            )

if result is None:
    print("No query backtest result in this run.")
else:
    display(pd.Series(result.metrics, name="value").to_frame())
    display(result.trades.tail(20))


## 12. Query Backtest Performance And Trade Attribution

In [ ]:
def plot_trade_analytics(trades: pd.DataFrame):
    fig, axes = plt.subplots(2, 2, figsize=(18, 10))
    if trades is None or trades.empty:
        for ax in axes.ravel():
            ax.set_axis_off()
        return fig

    by_idea = trades.groupby("idea_id")["realized_pnl"].agg(["count", "sum", "mean"]).sort_values("sum")
    by_idea["sum"].plot(kind="barh", ax=axes[0, 0], color=["tab:red" if x < 0 else "tab:green" for x in by_idea["sum"]])
    axes[0, 0].set_title("Realized PnL By Idea")

    if "exit_reason" in trades:
        trades["exit_reason"].value_counts().plot(kind="bar", ax=axes[0, 1], color="tab:blue")
        axes[0, 1].set_title("Exit Reason Mix")
        axes[0, 1].tick_params(axis="x", rotation=45)
    else:
        axes[0, 1].set_axis_off()

    if "holding_days" in trades:
        trades["holding_days"].plot(kind="hist", bins=20, ax=axes[1, 0], color="tab:gray", alpha=0.80)
        axes[1, 0].set_title("Holding Period Distribution")
    else:
        axes[1, 0].set_axis_off()

    if {"entry_zscore", "realized_pnl", "target_weight"}.issubset(trades.columns):
        size = trades["target_weight"].abs().fillna(0.0) * 240 + 30
        axes[1, 1].scatter(trades["entry_zscore"], trades["realized_pnl"], s=size, alpha=0.65)
        axes[1, 1].axhline(0.0, color="black", lw=1)
        axes[1, 1].axvline(0.0, color="black", lw=1)
        axes[1, 1].set_title("Entry Z-Score Vs Realized PnL")
        axes[1, 1].set_xlabel("Entry Z-Score")
        axes[1, 1].set_ylabel("Realized PnL")
    else:
        axes[1, 1].set_axis_off()

    fig.tight_layout()
    return fig

if result is not None:
    display(performance_stats(result.daily_pnl).to_frame("query_daily_pnl"))
    display(drawdown_episodes(result.daily_pnl))
    plot_performance_dashboard(result.daily_pnl, title="Query Backtest Normalized PnL")
    plot_trade_analytics(result.trades)
else:
    print("Run the query backtest section first.")


## 13. Built-In Query Tear Sheet

In [ ]:
if result is not None and PARAMS["run_query_tearsheet"] and create_query_backtest_tearsheet is not None:
    try:
        tearsheet = create_query_backtest_tearsheet(result.query_backtest)
        display(tearsheet.summary_frame)
        display(tearsheet.analytics.trade_summary_frame)
        fig = tearsheet.plot(backend="plotly" if HAS_PLOTLY else "matplotlib")
        display(fig)
    except Exception as exc:
        print(f"Query tear sheet failed: {exc}")
elif create_query_backtest_tearsheet is None:
    print("BT.query_tearsheet is unavailable in this environment.")
else:
    print("No query backtest result to tear-sheet.")


## 14. Interactive Overview

In [ ]:
def plot_interactive_overview(
    proxy_pnl: pd.Series,
    snapshots_df: pd.DataFrame,
    signals_df: pd.DataFrame,
    weights_panel: pd.DataFrame,
):
    if not HAS_PLOTLY:
        print("Plotly is unavailable.")
        return None

    clean = proxy_pnl.dropna()
    equity = clean.cumsum()
    dd = equity - equity.cummax()
    fig = make_subplots(
        rows=4,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.06,
        subplot_titles=("Signal Proxy Equity", "Drawdown", "Signal Count / Overlay", "Gross And Net Target Weight"),
    )
    fig.add_trace(go.Scatter(x=equity.index, y=equity, name="Proxy Equity", mode="lines"), row=1, col=1)
    fig.add_trace(go.Scatter(x=dd.index, y=dd, name="Drawdown", mode="lines", fill="tozeroy"), row=2, col=1)
    fig.add_trace(go.Scatter(x=snapshots_df.index, y=snapshots_df["n_signals"], name="Signal Count", mode="lines"), row=3, col=1)
    if "overlay_scale" in snapshots_df:
        fig.add_trace(go.Scatter(x=snapshots_df.index, y=snapshots_df["overlay_scale"], name="Overlay", mode="lines"), row=3, col=1)
    gross = weights_panel.abs().sum(axis=1)
    net = weights_panel.sum(axis=1)
    fig.add_trace(go.Scatter(x=gross.index, y=gross, name="Gross", mode="lines"), row=4, col=1)
    fig.add_trace(go.Scatter(x=net.index, y=net, name="Net", mode="lines"), row=4, col=1)
    fig.update_layout(template="plotly_white", height=900, title="Risk Premia Pairs PCA Interactive Overview")
    return fig

plot_interactive_overview(proxy_pnl, snapshots_df, signals_df, weights_panel)


## 15. Parameter Sweep

This uses the fast signal-level proxy, not the query engine. Turn on `PARAMS["run_param_sweep"]` when you want a broader sensitivity check.

In [ ]:
SWEEP_GRID = {
    "residual_z_entry": [1.0, 1.25, 1.50, 2.0],
    "pca_components": [1, 2, 3],
    "phase1_min_realized_sharpe": [-1.0, -0.5, 0.0],
    "l2_to_phase1": [1.0, 10.0, 25.0],
}


def run_signal_proxy_sweep(
    panel: pd.DataFrame,
    returns_panel: pd.DataFrame,
    base_config: RiskPremiaPairsPCAConfig,
    grid: dict[str, list[Any]],
) -> pd.DataFrame:
    rows = []
    keys = list(grid)
    for values in itertools.product(*(grid[key] for key in keys)):
        overrides = dict(zip(keys, values))
        cfg = replace(base_config, **overrides)
        table = build_risk_premia_pca_signal_table(panel, cfg, returns_panel=returns_panel)
        sdf, _ = signal_table_to_frames(table)
        w = build_weight_panel(sdf, panel.index, panel.columns)
        pnl = signal_proxy_pnl(w, returns_panel)
        stats = performance_stats(pnl)
        row = {**overrides, **stats.to_dict()}
        row["signal_rows"] = len(sdf)
        row["active_days"] = int((w.abs().sum(axis=1) > 0.0).sum())
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["sharpe", "total_pnl"], ascending=False).reset_index(drop=True)


if PARAMS["run_param_sweep"]:
    sweep_results = run_signal_proxy_sweep(panel, returns_panel, config, SWEEP_GRID)
    display(sweep_results.head(25))

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    sns.scatterplot(
        data=sweep_results,
        x="active_days",
        y="sharpe",
        hue="residual_z_entry",
        size="l2_to_phase1",
        ax=axes[0],
        palette="viridis",
    )
    axes[0].set_title("Sharpe Vs Active Days")

    pivot = sweep_results.pivot_table(index="pca_components", columns="residual_z_entry", values="sharpe", aggfunc="max")
    sns.heatmap(pivot, annot=True, fmt=".2f", cmap="vlag", center=0.0, ax=axes[1])
    axes[1].set_title("Best Proxy Sharpe By PCA Components And Entry Z")
    fig.tight_layout()
else:
    sweep_results = pd.DataFrame()
    print("Parameter sweep skipped. Set PARAMS['run_param_sweep'] = True to run it.")


## 16. Export Research Artifacts

In [ ]:
if PARAMS["export_outputs"]:
    output_dir = REPO_ROOT / "analysis_outputs" / "risk_premia_pairs_pca"
    output_dir.mkdir(parents=True, exist_ok=True)
    panel.to_csv(output_dir / "panel.csv")
    returns_panel.to_csv(output_dir / "returns_panel.csv")
    residual_panel.to_csv(output_dir / "pca_residuals.csv")
    zscore_panel.to_csv(output_dir / "pca_zscores.csv")
    pca_evr.to_csv(output_dir / "pca_explained_variance.csv")
    snapshots_df.to_csv(output_dir / "snapshots.csv")
    signals_df.to_csv(output_dir / "signals.csv", index=False)
    weights_panel.to_csv(output_dir / "target_weights.csv")
    proxy_pnl.to_csv(output_dir / "signal_proxy_pnl.csv")
    if result is not None:
        result.trades.to_csv(output_dir / "query_trades.csv", index=False)
        result.daily_pnl.to_csv(output_dir / "query_daily_pnl.csv")
    if not sweep_results.empty:
        sweep_results.to_csv(output_dir / "signal_proxy_sweep.csv", index=False)
    print(f"Wrote artifacts to {output_dir}")
else:
    print("Export skipped. Set PARAMS['export_outputs'] = True to write CSV artifacts.")


## 17. Live Execution Checklist

Before running this notebook against live or historical production data:

1. Confirm that each panel column is an executable idea id or has an explicit `query_factory` mapping.
2. Use swap-spread or basis packages for swap-spread RV; the default `make_irs_query_factory` only builds IRS outright, curve, and fly queries from ids like `2Y`, `2Y/10Y`, or `2Y/5Y/10Y`.
3. Keep the first live run small. Inspect signal counts, gross weight, overlay scale, and latest z-scores before running a long query backtest.
4. Treat signal proxy results as screening diagnostics. Use the query backtest result for execution PnL, trade lifecycle, MTM, and handler integration checks.
5. When a parameter sweep looks promising, rerun the focused tests and then validate the selected configuration with the query engine.